# 🧠 Reasoning Pruning: Interactive Exploration & Overthinking Discovery

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/avrymi-asraf/reasoning-pruning-agy/blob/master/notebooks/01_explore_pruning.ipynb)

This notebook implements the complete **Reasoning Pruning Exploration & Transition Extraction** workflow for analyzing reasoning trajectories, identifying overthinking patterns, and creating Pruning-Transition ($x \to y$) training datasets for models such as **Google Gemma** (`google/gemma-4-12B-it`, `google/gemma-2-9b-it`).

### 🎯 The Core Concept:
$$\text{Question } q \xrightarrow[\text{Generation}]{\text{Model } G} \text{Trace } (s_1..s_n) \xrightarrow[\text{Auditor } D]{\text{Find Skip}} \text{Skip } s_k \xrightarrow[\text{Extract}]{\text{Pair}} (x \to y) \xrightarrow[\text{Rollout}]{\text{Multi-Depth}} \text{PT Dataset}$$

| Module | Purpose |
|---|---|
| **1. Trace Generation** | Produce full step-by-step reasoning trajectories across task families. |
| **2. Overthinking Audit** | Detect redundant steps, conversational filler, and non-deductive loops. |
| **3. Visual Diffing** | Inline color-coded diffs (green prefix, red pruned steps, blue target jump). |
| **4. Recursive Rollout** | Multi-depth iterative pruning producing chained $(x \to y)$ training pairs. |
| **5. HF Dataset Export** | Package transitions into Hugging Face `Dataset` ready for SFT / QLoRA fine-tuning. |

## 1. Setup & Environment

Install and import `reasoning_pruning` along with evaluation and visualization dependencies.

In [ ]:
# 1. Detect environment and bootstrap if running in Google Colab
import os
import sys

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if IN_COLAB:
    print("🚀 Running on Google Colab. Setting up environment and repository...")
    REPO_DIR = "/content/reasoning-pruning-agy"
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/avrymi-asraf/reasoning-pruning-agy.git {REPO_DIR}
    %cd {REPO_DIR}
    !pip install -q -e .
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    print("✅ reasoning-pruning repository cloned and installed!")
else:
    print("💻 Running in local environment.")

import json
import re
import pandas as pd
from IPython.display import HTML, display

# Import core reasoning_pruning tools
import reasoning_pruning as rp
from reasoning_pruning.types import (
    ReasoningTrace,
    PruneDecision,
    TransitionExample,
    RolloutResult,
)

# Optional: Set API keys if using API-based models (OpenAI, Gemini, Anthropic, DeepSeek)
# os.environ["OPENAI_API_KEY"] = "your-api-key"
# os.environ["GEMINI_API_KEY"] = "your-api-key"
# os.environ["HF_TOKEN"] = "your-hf-token"

print(f"✅ reasoning_pruning v{rp.__version__} loaded successfully.")


## 2. Taxonomy of Overthinking in Reasoning Models

When reasoning models are prompted to *"think step by step"*, they frequently exhibit six distinct patterns of unnecessary and redundant computation:

| Pattern Archetype | Description & Cognitive Flaw | Overthinking Example | Pruning Action ($x \to y$) |
|---|---|---|---|
| **1. Conversational Preamble** | Meta-filler and introductory strategy statements. | *"Here's how to solve this step-by-step:"* | **Bypass directly** to first calculation. |
| **2. Question Restatement** | Paraphrasing prompt givens verbatim into separate steps. | *"Janet begins with 3 packs of 12 eggs..."* | **Compress into** direct calculation. |
| **3. Axiomatic / Definition Restatement** | Re-deriving elementary textbook definitions. | *"An even number is any integer divisible by 2..."* | **Skip directly** to checking the specific number. |
| **4. Redundant Verification Loops** | Double-checking arithmetic loops and backtracking. | *"Let me verify: 12 - 4 = 8, 8 + 4 = 12, yes."* | **Eliminate check** and proceed to final answer. |
| **5. Null Operation Narration** | Explicitly verbalizing identity actions that do not change state. | *"'Leaving the coin alone' means no change occurs."* | **Skip identity step** and retain state invariants. |
| **6. Encyclopedic Detour** | Emitting biological or geographic trivia not needed for deduction. | Explaining English Channel width & penguin bone buoyancy before answering *"No"*. | **Jump directly** to *"Penguins are flightless birds, so they cannot fly."* |

## 3. The 6-Family Reasoning Spectrum Benchmark Suite

We define a curated benchmark spectrum across 6 major cognitive task families to probe model behavior:

In [ ]:
TASK_SPECTRUM = [
    # 1. Arithmetic & Word Math
    {
        "task_id": "math_01",
        "category": "Arithmetic & Word Math",
        "question": "Janet buys 3 packs of 12 eggs. She bakes 2 cakes using 4 eggs each. How many eggs does she have left?",
        "ground_truth": "28",
        "description": "Multi-step arithmetic word problem with purchasing and subtraction",
    },
    {
        "task_id": "math_02",
        "category": "Arithmetic & Word Math",
        "question": "A baker has 45 cookies. He packages 5 cookies per box. How many boxes can he fill completely?",
        "ground_truth": "9",
        "description": "Direct single-operation division with simple numbers",
    },
    {
        "task_id": "math_03",
        "category": "Arithmetic & Word Math",
        "question": "The sum of two numbers is 20 and their difference is 4. What is the larger number?",
        "ground_truth": "12",
        "description": "Simple 2-variable mental algebra",
    },
    # 2. Commonsense & Physical Logic
    {
        "task_id": "common_01",
        "category": "Commonsense & Physical Logic",
        "question": "Can a penguin fly over the English Channel?",
        "ground_truth": "No",
        "description": "Species biological capability constraint check",
    },
    {
        "task_id": "common_02",
        "category": "Commonsense & Physical Logic",
        "question": "If you put a hot cup of coffee in a freezer at -18°C, will it become hotter or colder?",
        "ground_truth": "Colder",
        "description": "Basic thermodynamics commonsense",
    },
    {
        "task_id": "common_03",
        "category": "Commonsense & Physical Logic",
        "question": "Does an elephant need an umbrella when it rains in the savanna?",
        "ground_truth": "No",
        "description": "Physical & animal behavior logic",
    },
    # 3. Multi-hop & Deductive Logic
    {
        "task_id": "multihop_01",
        "category": "Multi-hop & Deductive Logic",
        "question": "Christopher Nolan directed 'Inception'. Which country was Christopher Nolan born in?",
        "ground_truth": "United Kingdom (London, England)",
        "description": "2-hop entity retrieval and deduction",
    },
    {
        "task_id": "multihop_02",
        "category": "Multi-hop & Deductive Logic",
        "question": "All roses are flowers. All flowers need sunlight to survive. Does a rose need sunlight to survive?",
        "ground_truth": "Yes",
        "description": "Elementary categorical syllogism",
    },
    {
        "task_id": "multihop_03",
        "category": "Multi-hop & Deductive Logic",
        "question": "To enter the tournament, a player must be at least 18 years old and rated above 1500. Alex is 22 years old with a rating of 1650. Is Alex eligible to enter?",
        "ground_truth": "Yes",
        "description": "Compound boolean condition verification",
    },
    # 4. Algorithmic & State Tracking
    {
        "task_id": "algo_01",
        "category": "Algorithmic & State Tracking",
        "question": "A coin starts heads up. You flip it once, then leave it alone, then flip it again. Is the coin currently heads or tails?",
        "ground_truth": "Heads",
        "description": "Parity state tracking with null operations",
    },
    {
        "task_id": "algo_02",
        "category": "Algorithmic & State Tracking",
        "question": "Today is Wednesday. What day of the week will it be exactly 3 days from now?",
        "ground_truth": "Saturday",
        "description": "Modular date arithmetic",
    },
    {
        "task_id": "algo_03",
        "category": "Algorithmic & State Tracking",
        "question": "You have 3 red balls and 4 blue balls in a box. You remove 1 red ball. How many total balls are left in the box?",
        "ground_truth": "6",
        "description": "Inventory state tracking with color property",
    },
    # 5. Extractive & Span QA
    {
        "task_id": "extract_01",
        "category": "Extractive & Span QA",
        "question": "Context: The Apollo 11 mission landed on the Moon on July 20, 1969, carrying Neil Armstrong and Buzz Aldrin. Question: In what year did Apollo 11 land on the Moon?",
        "ground_truth": "1969",
        "description": "Direct temporal span extraction from short context",
    },
    {
        "task_id": "extract_02",
        "category": "Extractive & Span QA",
        "question": "Context: Python was created by Guido van Rossum and first released in 1991. Question: Who created Python?",
        "ground_truth": "Guido van Rossum",
        "description": "Direct entity span extraction from short context",
    },
    {
        "task_id": "extract_03",
        "category": "Extractive & Span QA",
        "question": "Context: The headquarters of the International Olympic Committee is located in Lausanne, Switzerland. Question: In which city is the IOC headquartered?",
        "ground_truth": "Lausanne",
        "description": "Direct geographic location extraction",
    },
    # 6. Axiomatic & Math Properties
    {
        "task_id": "axiom_01",
        "category": "Axiomatic & Math Properties",
        "question": "Is 144 an even number?",
        "ground_truth": "Yes",
        "description": "Parity rule application",
    },
    {
        "task_id": "axiom_02",
        "category": "Axiomatic & Math Properties",
        "question": "Is 17 a prime number?",
        "ground_truth": "Yes",
        "description": "Prime divisibility check",
    },
    {
        "task_id": "axiom_03",
        "category": "Axiomatic & Math Properties",
        "question": "Which number is strictly greater: 0.7 or 0.07?",
        "ground_truth": "0.7",
        "description": "Decimal place value comparison",
    },
]

# Display task spectrum as clean DataFrame
df_spectrum = pd.DataFrame(TASK_SPECTRUM)[["task_id", "category", "question", "ground_truth"]]
display(df_spectrum.head(10))

## 4. Single-Trace Generation & Visual Trace Diffing

Let's select a task, inspect the generated trajectory, audit it for skippable steps, and render an **interactive HTML diff** with:
- 🟩 **Green text**: Kept context prefix ($x$)
- 🟥 **Red strikethrough**: Removable/redundant thoughts ($s_k$)
- 🟦 **Blue text**: Next useful deduction target ($y$)

In [ ]:
# 1. Select a benchmark task (e.g. Commonsense Physical Logic)
selected_task = TASK_SPECTRUM[3]  # Can a penguin fly over the English Channel?
question = selected_task["question"]

print(f"Task ID:   {selected_task['task_id']}")
print(f"Category:  {selected_task['category']}")
print(f"Question:  {question}")

# 2. Example generated reasoning trace (from Gemma reasoning)
trace = ReasoningTrace(
    question=question,
    steps=[
        "Here's the breakdown of why this is impossible:",
        "The English Channel is a 34-kilometer wide body of water separating England and France.",
        "Penguins are biologically flightless birds whose wings have evolved into flippers for underwater swimming.",
        "Because penguins lack the aerodynamic wing structure and lift required for aerial flight, they cannot fly across any distance.",
        "Final Answer: No, a penguin cannot fly over the English Channel."
    ],
    generator_model="google/gemma-4-12B-it",
    token_count=185,
)

# 3. Decision Auditor: Find first removable unit
decision = PruneDecision(
    can_skip=True,
    skip_start_idx=0,
    skip_end_idx=1,
    skipped_steps=[
        "Here's the breakdown of why this is impossible:",
        "The English Channel is a 34-kilometer wide body of water separating England and France."
    ],
    next_step="Penguins are biologically flightless birds whose wings have evolved into flippers for underwater swimming.",
    reason="Step 0 is conversational preamble and Step 1 is geographical trivia; flight capability is universally 0.",
    decision_model="decision-auditor-v1",
)

# 4. Render inline colored HTML diff
html_diff = rp.render_trace_diff(trace, decision, as_html=True)
display(HTML(html_diff))

## 5. Transition Extraction: Constructing the ($x \to y$) Training Pair

From the pruned trace and audit decision, we extract the core training instance:
- **Input $x$**: $\text{Question} + \text{Useful Prefix}$
- **Target $y$**: $\text{Next Useful Step}$ (bypassing the redundant block)

In [ ]:
transition = rp.extract_transition(
    trace=trace,
    decision=decision,
    depth=1,
    example_id=f"ex_{selected_task['task_id']}_d1",
)

print("=" * 60)
print(f"🎯 EXTRACTED PRUNING-TRANSITION EXAMPLE: {transition.id}")
print("=" * 60)
print(f"📌 Input Context (x):\n{transition.input_x}\n")
print(f"🚀 Target Continuation (y):\n{transition.target_y}\n")
print(f"✂️ Skipped Thoughts:\n{transition.skipped_steps}\n")
print(f"💡 Audit Justification:\n{transition.skip_reason}")

## 6. Multi-Depth Recursive Rollout ($x_1 \to y_1 \to x_2 \to y_2$)

Reasoning chains often contain multiple non-contiguous redundant blocks (e.g. preamble at start, variable re-declaration in middle, redundant check at end). **Recursive Rollout** prunes each layer iteratively to produce multi-depth transitions:

In [ ]:
# Demonstration on a 20-step algebra overthinking problem
algebra_q = "The sum of two numbers is 20 and their difference is 4. What is the larger number?"

raw_algebra_steps = [
    "Here's how to solve this problem step-by-step:",
    "Let 'x' represent the larger number and 'y' represent the smaller number.",
    "Set up the system of equations: x + y = 20 and x - y = 4.",
    "Add the two equations together: (x + y) + (x - y) = 20 + 4, which simplifies to 2x = 24.",
    "Divide both sides by 2: x = 12.",
    "Substitute back to verify: 12 + 8 = 20 and 12 - 8 = 4 (both hold).",
    "Therefore, the larger number is 12."
]

# Multi-Depth Rollout Jumps
# Multi-Depth Rollout Jumps (Strict Verbatim Sentence Omission)
rollout_transitions = [
    TransitionExample(
        id="math_03_d1",
        question=algebra_q,
        input_x=f"Question: {algebra_q}",
        target_y=raw_algebra_steps[2],  # Direct verbatim slice from model trace: "Set up the system..."
        depth=1,
        skipped_steps=[raw_algebra_steps[0], raw_algebra_steps[1]],  # Omit preamble + variable declaration
        skip_reason="Omit conversational preamble and trivial variable declarations; jump directly to equation setup.",
    ),
    TransitionExample(
        id="math_03_d2",
        question=algebra_q,
        input_x=f"Question: {algebra_q}

{raw_algebra_steps[2]}",
        target_y=raw_algebra_steps[4],  # Direct verbatim slice: "Divide both sides by 2: x = 12."
        depth=2,
        skipped_steps=[raw_algebra_steps[3]],  # Omit mechanical manipulation sentence
        skip_reason="Omit mechanical step narration; jump directly to simplified evaluation.",
    ),
    TransitionExample(
        id="math_03_d3",
        question=algebra_q,
        input_x=f"Question: {algebra_q}

{raw_algebra_steps[2]} {raw_algebra_steps[4]}",
        target_y=raw_algebra_steps[6],  # Direct verbatim slice: "Therefore, the larger number is 12." (Final Answer)
        depth=3,
        skipped_steps=[raw_algebra_steps[5]],  # Omit redundant double-check loop
        skip_reason="Omit redundant verification check; jump directly to identical final answer.",
    )
]

print("=" * 70)
print("🔬 MULTI-DEPTH ROLLOUT TRANSITIONS (STRICT SENTENCE OMISSION)")
print("=" * 70)
for t in rollout_transitions:
    print(f"
[Depth {t.depth}] ({t.id})")
    print(f"  Input Context (x): {t.input_x[:70]}...")
    print(f"  Target Step (y):   {t.target_y}")
    print(f"  Omitted Step(s):   {t.skipped_steps}")
    print(f"  Audit Reason:      {t.skip_reason}")


## 7. Full Reasoning Spectrum Audit & Empirical Overthinking Analysis

Let's load the benchmark sweep across all 18 tasks and evaluate which reasoning families exhibit the most excessive overthinking:

In [ ]:
# Load empirical exploration findings (generated across the 18 tasks on GPU)
data_file = "../data/exploration_results.json" if os.path.exists("../data/exploration_results.json") else "data/exploration_results.json"

if os.path.exists(data_file):
    with open(data_file) as f:
        results_data = json.load(f)
        
    cat_summary = results_data["category_summary"]
    
    summary_rows = []
    for cat, stats in cat_summary.items():
        summary_rows.append({
            "Reasoning Family": cat,
            "Tasks Audited": stats["total_tasks"],
            "Overthinking Rate (%)": f"{stats['overthinking_rate_pct']:.1f}%",
            "Avg Original Steps": f"{stats['avg_steps_per_task']:.1f}",
            "Avg Generated Tokens": f"{stats['avg_tokens_per_task']:.1f}",
            "Step Reduction (%)": f"{stats['step_reduction_pct']:.1f}%",
        })
        
    df_summary = pd.DataFrame(summary_rows)
    print("📊 OVERTHINKING AUDIT BY REASONING FAMILY:")
    display(df_summary)
else:
    print("Exploration results file not found locally. Run scripts/explore_overthinking_patterns.py to generate live data.")

## 8. Hugging Face Dataset Formatting & Hub Synchronization

Finally, we assemble all extracted $(x \to y)$ transitions into a Hugging Face `Dataset` ready for 4-bit QLoRA SFT training (`rp train`).

In [ ]:
from datasets import Dataset

# Prepare transition dataset rows
dataset_rows = []
for t in rollout_transitions:
    dataset_rows.append({
        "id": t.id,
        "question": t.question,
        "input_x": t.input_x,
        "target_y": t.target_y,
        "depth": t.depth,
        "skipped_steps": json.dumps(t.skipped_steps),
        "skip_reason": t.skip_reason,
        "generator_model": "google/gemma-4-12B-it",
    })

hf_dataset = Dataset.from_list(dataset_rows)
print(f"✅ Successfully built Hugging Face Dataset with {len(hf_dataset)} transition rows:")
display(hf_dataset.to_pandas().head())

# To push to Hugging Face Hub, uncomment:
# hf_dataset.push_to_hub("your-username/rp-gemma-transitions-v1", private=True)